## Simplifying the route polyline

The [Routing API notebook](obm_9_routing_api.ipynb) ended on a one-liner:

```python
search_polyline = encode(simplify_polyline(decode(section.polyline), 2000))
```

This notebook opens that black box. Simplification is needed because the polyline plays
**two different roles**, with two contradictory requirements:

| role | consumer | wants |
| --- | --- | --- |
| drawing the route | the map | every point, so that the line sticks to the road |
| searching along the route | `route=<polyline>;w=<width>` | at most 2000 points |

The second one is not a preference: the Search API rejects a `route` parameter holding more
points than that, and a route of a few hundred kilometers is returned with several thousand.
So an application that searches along a route it did not simplify works on short routes and
starts failing on long ones — which is the kind of bug that reaches production.

In [ ]:
from flexpolyline import decode, encode

from here_search_demo.auth import Credentials
from here_search_demo.http import HTTPSession
from here_search_demo.routing_api import RoutingAPI

credentials = Credentials()
routing_api = RoutingAPI(credentials=credentials)
session = HTTPSession()

start_pos = 48.57436, 7.71556  # Strasbourg, where the trip started
at_pos = 48.03529, 7.37121  # near Colmar, where the driver is now
stop_pos = 45.75049, 4.84418  # Lyon, the destination

route = await routing_api.routes(session=session, origin=start_pos, destination=stop_pos)
(section,) = RoutingAPI.sections(route)

waypoints = decode(section.polyline)
print(f"{round(section.length / 1000)} km, {round(section.duration / 60)} min")
print(f"{len(waypoints)} points, {len(section.polyline)} polyline characters")

Roughly one point every hundred meters. The ceiling forbids sending them all —
`API` is instantiated with `raise_for_status=False` here so that the API error surfaces as
data rather than as an exception:

In [ ]:
from here_search_demo.api import API

api = API(credentials=credentials, raise_for_status=False)


async def corridor_search(points, width=500):
    """Browse restaurants inside a *width*-meter corridor around *points*."""
    return await api.browse(
        session=session,
        latitude=at_pos[0],
        longitude=at_pos[1],
        categories=["100-1000"],  # restaurants
        polyline=encode(points),
        width=width,
        limit=20,
        lang="en",
    )


rejected = await corridor_search(waypoints)
rejected.data["title"], rejected.data["status"]

### Douglas–Peucker, driven by a point budget

[Ramer–Douglas–Peucker][1] is the classic polyline simplification: it keeps the two
endpoints, finds the point furthest away from the segment joining them, and recurses on
both halves as long as that distance exceeds a tolerance `eps`. Shapely exposes the GEOS
implementation as `shapely.simplify(geometry, eps)`, so nothing has to be written by hand
— except that it takes a *tolerance*, while what is needed here is a *maximum number of
points*.

The two are related monotonically: the larger `eps`, the fewer points. A binary search on
`eps`, between `0` and the size of the bounding box, converges towards the smallest
tolerance that fits the budget. The bounding box is a valid upper bound because the
tolerance is expressed in the unit of the coordinates — **degrees** here — and no
point-to-segment distance of the route can exceed it.

`preserve_topology=False` selects the plain Douglas–Peucker variant; the topology-checking
one is much slower and guards against self-intersections that a road route does not have.

[1]: https://en.wikipedia.org/wiki/Ramer%E2%80%93Douglas%E2%80%93Peucker_algorithm

In [ ]:
from shapely import LineString, get_coordinates, get_num_coordinates
from shapely import simplify as shapely_simplify


def simplify_to(points, max_points, iterations=15):
    """Return at most *max_points* of *points*, keeping the shape of the polyline."""
    if len(points) <= max_points:
        return points
    if max_points < 2:
        raise ValueError("max_points must be >= 2")

    line = LineString(points)
    lats = [lat for lat, _ in points]
    lons = [lon for _, lon in points]
    eps_low, eps_high = 0.0, max(max(lats) - min(lats), max(lons) - min(lons))

    best = None
    for _ in range(iterations):
        eps = (eps_low + eps_high) / 2
        candidate = shapely_simplify(line, eps, preserve_topology=False)
        if get_num_coordinates(candidate) <= max_points:
            best, eps_high = candidate, eps  # fits: try to keep even fewer points
        else:
            eps_low = eps  # too many points: raise the tolerance

    return points if best is None else [tuple(row) for row in get_coordinates(best)]


[len(simplify_to(waypoints, 2000, iterations)) for iterations in (10, 15, 20, 25, 30)]

The iteration count is a *precision* knob, and the search is deliberately one-sided: the
result is always at or under the budget, never over it. Ten iterations use 7% of the
budget; the default fifteen a bit more than half of it; twenty-five lands on it exactly.
Under-using the budget is safe but gives away accuracy for free, so the default is worth
revisiting in an application that routinely handles long routes.

### What is lost

The quantity to watch is not the point count but the **deviation**: how far the simplified
line strays from the real one, in meters. Projecting both onto a local tangent plane turns
degrees into meters, and Shapely's `hausdorff_distance` then returns the worst of those
distances in a single call.

In [ ]:
import math

EARTH_RADIUS_M = 6_371_000.0


def to_local(points, ref_lat, ref_lon):
    """Project (lat, lon) degrees onto the tangent plane at the reference point, in meters."""
    cos_lat = math.cos(math.radians(ref_lat))
    return [
        (
            EARTH_RADIUS_M * math.radians(lon - ref_lon) * cos_lat,
            EARTH_RADIUS_M * math.radians(lat - ref_lat),
        )
        for lat, lon in points
    ]


ref_lat, ref_lon = waypoints[len(waypoints) // 2]
original_line = LineString(to_local(waypoints, ref_lat, ref_lon))

budgets = {}
for max_points in (2000, 1000, 500, 200, 50):
    simplified = simplify_to(waypoints, max_points, iterations=25)
    budgets[max_points] = simplified
    deviation = original_line.hausdorff_distance(LineString(to_local(simplified, ref_lat, ref_lon)))
    print(f"{max_points:>5} points budget -> {len(simplified):>5} points, "
          f"{len(encode(simplified)):>6} polyline characters, {round(deviation):>5} m deviation")

At the 2000-point ceiling the line stays within a few meters of the road, two orders of
magnitude below the 500 m corridor half-width: the simplification is invisible to the
search. It stops being invisible in the low hundreds of points: the deviation
approaches the corridor half-width, the line starts cutting corners, and it drags the
corridor off the road with it.

The demo ships the same function as
`widgets.route_geometry.simplify_polyline(points, max_points, iterations=15)`:

In [ ]:
from here_search_demo.widgets.route_geometry import simplify_polyline

simplify_to(waypoints, 2000) == simplify_polyline(waypoints, 2000)

### The effect on the search

The same `browse()` call is now repeated with three budgets. The 2000-point corridor is the
reference; the coarser ones lose results and gain others, because the corridor no longer
covers the same roads.

In [ ]:
reference = (await corridor_search(budgets[2000])).titles

for max_points in (2000, 200, 50):
    titles = (await corridor_search(budgets[max_points])).titles
    print(f"{max_points:>5} points -> {len(titles):>2} results, "
          f"{len(set(reference) & set(titles)):>2} shared with the reference")

### Seeing it

The lines are indistinguishable at country scale, so the second map below zooms on the
point where the 50-point version strays the most.

In [ ]:
from shapely import Point

coarse_line = LineString(to_local(budgets[50], ref_lat, ref_lon))
local_points = to_local(waypoints, ref_lat, ref_lon)
worst_index = max(range(len(waypoints)), key=lambda index: coarse_line.distance(Point(local_points[index])))

worst_position = waypoints[worst_index]
worst_position, round(coarse_line.distance(Point(local_points[worst_index])))

In [ ]:
from ipyleaflet import AwesomeIcon, Map, Marker, Polyline, TileLayer

def here_tile_layer():
    """Return a HERE Map Tile API v3 raster layer, as `widgets.input_map.PositionMap` does."""
    return TileLayer(
        url=(
            "https://maps.hereapi.com/v3/base/mc/{z}/{x}/{y}/png"
            f"?style=lite.day&ppi=400&size=512&apiKey={credentials.api_key}&lang=en"
        ),
        attribution="© HERE 2026",
    )


def comparison_map(center, zoom):
    """Return a map showing the full polyline in blue and the 50-point one in orange."""
    map_widget = Map(
        center=center,
        zoom=zoom,
        basemap={},
        layout={"height": "500px"},
        zoom_control=False,
        scroll_wheel_zoom=True,
        double_click_zoom=False,
    )
    map_widget.add(here_tile_layer())
    map_widget.add(Polyline(locations=waypoints, color="#2c7fb8", weight=4, fill=False))
    map_widget.add(Polyline(locations=budgets[50], color="#e6550d", weight=2, fill=False))
    return map_widget


overview = comparison_map(((start_pos[0] + stop_pos[0]) / 2, (start_pos[1] + stop_pos[1]) / 2), 7)
overview.add(Marker(location=start_pos, title="start", draggable=False, icon=AwesomeIcon(name="play")))
overview.add(Marker(location=stop_pos, title="stop", draggable=False, icon=AwesomeIcon(name="flag")))
display(overview)

Zoomed on the worst point, the orange line leaves the road entirely — and a 500 m corridor
built around it would miss every place along the real one:

In [ ]:
display(comparison_map(worst_position, 12))

### In the demo application

`RouteEngine.build_flexpolyline()` does the split in three lines: `route_flexpolyline` is
the encoded full geometry, used by `widgets.route.RouteController` to draw the route, and
`search_flexpolyline` is the simplified one, passed to `API.browse()`/`API.discover()` as
the `polyline` argument. The ceiling is the `RouteEngine.max_waypoints_count` class
attribute, so an application that prefers another budget only overrides it.

The next notebook moves the *anchor* of that search: [a few minutes ahead of the
driver](obm_11_position_ahead.ipynb) rather than at the driver.

<!--
from urllib.parse import quote, unquote
svg = """<svg width="40" height="48" viewBox="0 0 40 48" fill="none" xmlns="http://www.w3.org/2000/svg"><path class="triangle" d="M11,36.8l-5.5,5.5L0,36.8H11z" fill="#48DAD0"></path><path class="HERE" d="M19.1,24.2c-1.2-1.4-1.1-2.1-0.4-2.8c0.9-0.9,1.7-0.5,2.7,0.5L19.1,24.2z M30.9,9.2c0.9-0.9,1.7-0.5,2.7,0.5
    L31.3,12C30.1,10.7,30.2,9.9,30.9,9.2z M38,11.6c-1.1,1.6-2.9,4.2-4.9,2.2l5-5c-0.4-0.5-0.8-0.9-1-1.1c-2.7-2.7-5.7-2.7-8-0.4
	c-1.6,1.6-2,3.4-1.5,5.1l-1.6-1.8c-0.5,0.3-2.4,1.9-0.9,4.5l-1.8-1.5l-2.4,2.4l3.2,3.2c-2.5-1.9-5.1-1.8-7.2,0.4
	c-2.3,2.3-2.1,5-0.4,7.3l-0.3-0.3c-2.3-2.3-4.7-1.5-5.9-0.3c-0.9,0.9-1.5,2.2-1.3,3.1L4,24.6l-2.6,2.6l9.6,9.6h5.2l-3.5-3.5
	c-1.8-1.8-1.8-2.8-1-3.7c0.8-0.8,1.8-0.3,3.6,1.4l3.4,3.4l2.6-2.6l-3.2-3.2c2.3,1.8,5.1,1.9,7.7-0.6l0,0c1.5-1.4,2-2.8,2-2.8
	l-1.9-1.3c-1.1,1.6-2.9,4.2-4.9,2.3l5-5l3.1,3.1l2.7-2.7l-3.9-3.9c-1.8-1.8-0.7-3.5,0-4.1c0.4,0.7,0.9,1.5,1.5,2.1
	c2.5,2.5,5.7,3,8.6,0.2l0,0c1.5-1.4,2-2.8,2-2.8S38,11.6,38,11.6z" fill="#000000"></path></svg>
"""
print(f"""![HERE](data:image/svg+xml,{quote(svg)})
<span style="float:right; width:90%;"><sub><b>Copyright (c) 2020-2026 HERE Global B.V. and its affiliate(s). All rights reserved.</b>
This software, including documentation, is protected by copyright controlled by HERE. All rights are reserved. Copying, including reproducing, 
storing, adapting or translating, any or all of this material requires the prior written consent of HERE. This material also contains confidential 
information which may not be disclosed to others without the prior written consent of HERE.</sub></span>""")
-->
![HERE](data:image/svg+xml,%3Csvg%20width%3D%2240%22%20height%3D%2248%22%20viewBox%3D%220%200%2040%2048%22%20fill%3D%22none%22%20xmlns%3D%22http%3A//www.w3.org/2000/svg%22%3E%3Cpath%20class%3D%22triangle%22%20d%3D%22M11%2C36.8l-5.5%2C5.5L0%2C36.8H11z%22%20fill%3D%22%2348DAD0%22%3E%3C/path%3E%3Cpath%20class%3D%22HERE%22%20d%3D%22M19.1%2C24.2c-1.2-1.4-1.1-2.1-0.4-2.8c0.9-0.9%2C1.7-0.5%2C2.7%2C0.5L19.1%2C24.2z%20M30.9%2C9.2c0.9-0.9%2C1.7-0.5%2C2.7%2C0.5%0A%20%20%20%20L31.3%2C12C30.1%2C10.7%2C30.2%2C9.9%2C30.9%2C9.2z%20M38%2C11.6c-1.1%2C1.6-2.9%2C4.2-4.9%2C2.2l5-5c-0.4-0.5-0.8-0.9-1-1.1c-2.7-2.7-5.7-2.7-8-0.4%0A%09c-1.6%2C1.6-2%2C3.4-1.5%2C5.1l-1.6-1.8c-0.5%2C0.3-2.4%2C1.9-0.9%2C4.5l-1.8-1.5l-2.4%2C2.4l3.2%2C3.2c-2.5-1.9-5.1-1.8-7.2%2C0.4%0A%09c-2.3%2C2.3-2.1%2C5-0.4%2C7.3l-0.3-0.3c-2.3-2.3-4.7-1.5-5.9-0.3c-0.9%2C0.9-1.5%2C2.2-1.3%2C3.1L4%2C24.6l-2.6%2C2.6l9.6%2C9.6h5.2l-3.5-3.5%0A%09c-1.8-1.8-1.8-2.8-1-3.7c0.8-0.8%2C1.8-0.3%2C3.6%2C1.4l3.4%2C3.4l2.6-2.6l-3.2-3.2c2.3%2C1.8%2C5.1%2C1.9%2C7.7-0.6l0%2C0c1.5-1.4%2C2-2.8%2C2-2.8%0A%09l-1.9-1.3c-1.1%2C1.6-2.9%2C4.2-4.9%2C2.3l5-5l3.1%2C3.1l2.7-2.7l-3.9-3.9c-1.8-1.8-0.7-3.5%2C0-4.1c0.4%2C0.7%2C0.9%2C1.5%2C1.5%2C2.1%0A%09c2.5%2C2.5%2C5.7%2C3%2C8.6%2C0.2l0%2C0c1.5-1.4%2C2-2.8%2C2-2.8S38%2C11.6%2C38%2C11.6z%22%20fill%3D%22%23000000%22%3E%3C/path%3E%3C/svg%3E%0A)
<span style="float:right; width:90%;"><sub><b>Copyright (c) 2020-2026 HERE Global B.V. and its affiliate(s). All rights reserved.</b>
This software, including documentation, is protected by copyright controlled by HERE. All rights are reserved. Copying, including reproducing, 
storing, adapting or translating, any or all of this material requires the prior written consent of HERE. This material also contains confidential 
information which may not be disclosed to others without the prior written consent of HERE.</sub></span>